# 01 · Dal modello all'agente

Costruiamo, un pezzo alla volta, la scala che porta da una semplice chiamata al modello
fino a un **agente** che decide da solo quando usare uno strumento.

Tappe:
1. chiamata diretta al modello;
2. una *chain* (catena) con prompt e parser;
3. un **agente** con un tool e il suo loop ReAct;
4. ispezione del loop per vedere cosa succede davvero.

## Obiettivi, prerequisiti e modalità di lettura

Passerai da una singola inferenza a una chain e infine a un agente con tool. Durata indicativa: 25–35 minuti. Le risposte del modello non sono deterministiche; gli output strutturali sono invece prevedibili.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Caricamento della configurazione

Il blocco cerca `.env` risalendo dalla directory corrente. In questo modo il notebook funziona sia dalla radice sia da `notebooks/`, senza importare configurazione dal progetto.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Una riga simile a `Ambiente caricato da: .../.env`. Se il file o la chiave mancano, compare un errore esplicito prima di qualsiasi chiamata API.

### Spiegazione del blocco · Creazione del modello

`ChatOpenAI` adatta il provider all'interfaccia LangChain. `store=False` evita la conservazione server-side e il nome modello resta configurabile tramite ambiente.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

`Modello pronto: <nome-modello>`. Non viene ancora effettuata alcuna chiamata al modello.

## 1 · Chiamata diretta

Il livello più basso: mandiamo dei messaggi e riceviamo una risposta. Nessun prompt
riutilizzabile, nessuno strumento, nessun ciclo. Solo input → output.

### Spiegazione del blocco · Costruzione dei messaggi

Qui si separano istruzioni stabili (`SystemMessage`) e richiesta dell'utente (`HumanMessage`). Il blocco prepara dati, ma non invoca ancora il provider.

In [ ]:
# I messaggi sono tipizzati: uno di "sistema" (le istruzioni) e uno "umano" (la domanda).
from langchain_core.messages import HumanMessage, SystemMessage

messaggi = [
    SystemMessage(content="Sei un docente di sistemi agentici. Rispondi in italiano, breve."),
    HumanMessage(content="In una frase: che differenza c'è tra un modello e un agente?"),
]

### Output atteso

Nessun testo stampato. La variabile `messaggi` contiene esattamente due oggetti tipizzati.

### Spiegazione del blocco · Prima invocazione

`invoke` esegue una singola inferenza. La risposta è un `AIMessage`, non una semplice stringa: `.text` ne estrae la parte leggibile.

In [ ]:
# `invoke` fa una singola chiamata e restituisce un AIMessage.
risposta = model.invoke(messaggi)
print(risposta.text)   # `.text` estrae solo il testo della risposta

### Output atteso

Una frase in italiano che distingue modello e agente. La formulazione varia tra esecuzioni, mentre il significato dovrebbe restare equivalente.

La risposta è un oggetto `AIMessage`. Contiene il testo ma anche metadati utili, come il
conteggio dei token. Diamoci un'occhiata.

### Spiegazione del blocco · Ispezione dei metadati

Il risultato conserva tipo e usage. Questo dato permette al runtime di misurare costo e budget senza stimare tutto dal testo.

In [ ]:
print("Tipo:", type(risposta).__name__)
print("Token usati:", risposta.usage_metadata)   # input/output/totale

### Output atteso

`Tipo: AIMessage` seguito da un dizionario con token di input, output e totale. I numeri dipendono dal modello.

## 2 · Una chain con LCEL

Una *chain* mette in fila più pezzi con l'operatore `|` (LangChain Expression Language).
Qui: un **prompt** con un segnaposto → il **modello** → un **parser** che tiene solo la stringa.

### Spiegazione del blocco · Prompt parametrico

`ChatPromptTemplate` trasforma un modello di messaggi in un componente riutilizzabile. `{argomento}` viene sostituito solo al momento dell'invocazione.

In [ ]:
# Il prompt ha una variabile {argomento}: potremo riusarlo con input diversi.
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sei un tutor di Python. Spiega con parole semplici e un mini esempio."),
    ("human", "Spiega: {argomento}"),
])

### Output atteso

Nessun output. `prompt` contiene due template di messaggio e richiede la variabile `argomento`.

### Spiegazione del blocco · Composizione LCEL

L'operatore `|` compone prompt, modello e parser. Il risultato è una pipeline invocabile: l'output del componente precedente diventa input del successivo.

In [ ]:
# Il parser trasforma l'AIMessage in una semplice stringa.
from langchain_core.output_parsers import StrOutputParser

# `|` compone: prompt -> model -> parser. Il risultato è ancora un oggetto "eseguibile".
chain = prompt | model | StrOutputParser()

### Output atteso

Nessun output. `chain` è pronta e restituirà direttamente una stringa anziché un `AIMessage`.

### Spiegazione del blocco · Esecuzione della chain

Il dizionario riempie il parametro del prompt. La stessa chain può essere riusata con argomenti diversi senza duplicare istruzioni.

In [ ]:
# Invochiamo la chain riempiendo il segnaposto {argomento}.
print(chain.invoke({"argomento": "una list comprehension"}))

### Output atteso

Una spiegazione semplice della list comprehension con un piccolo esempio Python. Testo e codice possono variare.

## 3 · Un agente con un tool

Un **agente** è un modello che può chiamare strumenti in un ciclo, finché non ha finito.
Trasformiamo una funzione Python in tool con `@tool`: i *tipi* e la *docstring* diventano
lo "schema" che il modello legge per capire quando e come usarlo.

### Spiegazione del blocco · Definizione di un tool

Il decoratore converte la funzione in uno strumento. Tipi e docstring diventano schema che il modello usa per decidere quando e come chiamarlo; la validazione positiva impedisce misure senza senso.

In [ ]:
# La docstring spiega al modello COSA fa il tool; i tipi dicono quali argomenti servono.
from langchain_core.tools import tool


@tool
def area_rettangolo(base: float, altezza: float) -> float:
    """Calcola l'area di un rettangolo date base e altezza in metri."""
    if base <= 0 or altezza <= 0:
        raise ValueError("Le misure devono essere positive.")
    return base * altezza

### Output atteso

Nessun output. `area_rettangolo` espone gli argomenti numerici `base` e `altezza`.

### Spiegazione del blocco · Creazione del loop agente

`create_agent` collega modello e tool. Il system prompt impone di usare il calcolo verificabile invece di produrre un numero soltanto per generazione linguistica.

In [ ]:
# `create_agent` costruisce il loop modello -> tool -> osservazione -> modello (pattern ReAct).
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[area_rettangolo],
    system_prompt="Usa sempre il tool per i calcoli, non stimare a mente.",
)

### Output atteso

Nessun output. `agente` è un graph invocabile con stato `messages`.

### Spiegazione del blocco · Richiesta che richiede un tool

La domanda contiene valori compatibili con lo schema. Il modello dovrebbe emettere una tool call, ricevere `100.0` e solo dopo formulare la risposta finale.

In [ ]:
# L'agente riceve i messaggi in un dizionario e restituisce lo stato finale.
esito = agente.invoke({
    "messages": [{"role": "user", "content": "Quanto misura l'area di 12.5 m per 8 m?"}]
})
print(esito["messages"][-1].text)   # l'ultimo messaggio è la risposta finale

### Output atteso

Una risposta che indica area pari a `100 m²`. La frase varia, ma il valore deve provenire dal tool.

## 4 · Guardare dentro il loop

La risposta finale nasconde i passaggi. La lista `messages` invece li mostra tutti:
la richiesta del tool da parte del modello, il risultato del tool, e la conclusione.

### Spiegazione del blocco · Trace del ciclo ReAct

La lista finale dei messaggi espone il ciclo completo: input umano, richiesta del modello, osservazione del tool e risposta conclusiva. È il modo più semplice per non trattare l'agente come scatola nera.

In [ ]:
# Stampiamo la sequenza di messaggi per vedere il ragionamento in azione.
for i, m in enumerate(esito["messages"]):
    tipo = type(m).__name__
    if getattr(m, "tool_calls", None):        # il modello ha CHIESTO di usare un tool
        print(f"{i}. {tipo}: chiama {m.tool_calls[0]['name']} con {m.tool_calls[0]['args']}")
    elif tipo == "ToolMessage":               # il RISULTATO del tool
        print(f"{i}. {tipo}: risultato = {m.content}")
    else:
        print(f"{i}. {tipo}: {str(m.content)[:80]}")

### Output atteso

Quattro passaggi circa: `HumanMessage`, `AIMessage` con `area_rettangolo`, `ToolMessage` con `100.0`, quindi `AIMessage` finale.

## Prova tu

- Aggiungi un tool `perimetro_rettangolo` e chiedi entrambi i valori.
- Rendi vaga la docstring del tool e osserva come il modello sbaglia a sceglierlo.

**Idea chiave**: una chain segue un percorso fisso deciso da te; un agente sceglie il
prossimo passo da solo, entro i limiti (i tool) che gli dai.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: testare il tool senza modello

### Spiegazione del blocco

Prima di attribuire un errore all'agente, conviene verificare direttamente funzione e validazione degli argomenti.

In [ ]:
for misure in ({"base": 3, "altezza": 2}, {"base": -1, "altezza": 4}):
    try:
        print(misure, "->", area_rettangolo.invoke(misure))
    except ValueError as errore:
        print(misure, "-> errore:", errore)

### Output atteso

Il primo caso stampa `6.0`; il secondo mostra l'errore sulle misure positive.

## Esempio aggiuntivo: osservare il prompt prima della chiamata

### Spiegazione del blocco

Formattare il prompt senza invocare il modello permette di controllare esattamente quali messaggi verranno inviati.

In [ ]:
anteprima = prompt.format_messages(argomento="un dizionario Python")
for messaggio in anteprima:
    print(type(messaggio).__name__, "->", messaggio.content)

### Output atteso

Due righe: istruzione del tutor e richiesta con `un dizionario Python`. Nessuna chiamata API aggiuntiva.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.